# Time Series Analysis – Lab Experiments 1–10


## Experiment 1


In [ ]:
%pip install pandas numpy scipy statsmodels matplotlib scikit-learn notebook


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import boxcox, ks_2samp
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.filters.hp_filter import hpfilter
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox

df = pd.read_csv('Microsoft_Stock.csv')
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
close = df['Close'].astype(float)

display(df.head())
df.info()


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df['Close'])
plt.title('Microsoft Stock Price')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.tight_layout()
plt.show()


## Experiment 2


In [ ]:
rm_90 = close.rolling(90).mean()
rm_365 = close.rolling(365).mean()

plt.figure(figsize=(10, 4))
close.plot(label='Close', alpha=0.5)
rm_90.plot(label='90-day mean')
rm_365.plot(label='365-day mean')
plt.legend()
plt.title('Trend of Microsoft Stock')
plt.show()


In [ ]:
m = close.to_frame('Close')
m['Month'] = m.index.month

m.boxplot(column='Close', by='Month', figsize=(10, 4))
plt.title('Monthly Boxplot')
plt.suptitle('')
plt.show()

m.groupby('Month')['Close'].mean().plot(kind='bar', figsize=(10, 4))
plt.title('Monthly Mean Close Price')
plt.xlabel('Month')
plt.show()


In [ ]:
monthly = close.resample('ME').mean()

cycle, trend = hpfilter(monthly, lamb=129600)

fig, ax = plt.subplots(2, figsize=(10, 6), sharex=True)
trend.plot(ax=ax[0], title='HP Trend')
cycle.plot(ax=ax[1], title='Cyclic Component')
plt.tight_layout()
plt.show()


## Experiment 3


In [ ]:
add = seasonal_decompose(monthly, model='additive', period=12)
add.plot()
plt.show()


In [ ]:
mul = seasonal_decompose(monthly, model='multiplicative', period=12)
mul.plot()
plt.show()


In [ ]:
print('Additive residual standard deviation:', add.resid.std())
print('Multiplicative residual standard deviation:', mul.resid.std())

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
add.resid.plot(ax=ax[0], title='Additive Residual')
mul.resid.plot(ax=ax[1], title='Multiplicative Residual')
plt.tight_layout()
plt.show()


## Experiment 4


In [ ]:
rm = close.rolling(30).mean()
rs = close.rolling(30).std()

plt.figure(figsize=(10, 4))
plt.plot(close, label='Original')
plt.plot(rm, label='Rolling Mean')
plt.plot(rs, label='Rolling Std')
plt.legend()
plt.title('Rolling Statistics')
plt.show()


In [ ]:
adf_result = adfuller(close.dropna(), autolag='AIC')
print('ADF Statistic:', adf_result[0])
print('p-value:', adf_result[1])
print('Critical values:', adf_result[4])


In [ ]:
kpss_result = kpss(close.dropna(), regression='c', nlags='auto')
print('KPSS Statistic:', kpss_result[0])
print('p-value:', kpss_result[1])
print('Critical values:', kpss_result[3])


## Experiment 5


In [ ]:
close.rolling(30).mean().plot(label='Rolling Mean', figsize=(10, 4))
close.rolling(30).std().plot(label='Rolling Std')
plt.legend()
plt.show()


In [ ]:
h = len(close) // 2
stat, p = ks_2samp(close.iloc[:h], close.iloc[h:])
print('KS statistic:', stat)
print('p-value:', p)


In [ ]:
d_ks = close.diff().dropna()
h = len(d_ks) // 2
result = ks_2samp(d_ks.iloc[:h], d_ks.iloc[h:])
print('KS statistic:', result.statistic)
print('p-value:', result.pvalue)


## Experiment 6


In [ ]:
log_s = np.log(close)
sq_s = np.square(close)
bc_values, lam = boxcox(close)
bc_s = pd.Series(bc_values, index=close.index)
diff_s = close.diff().dropna()
log_diff = log_s.diff().dropna()
detrended = log_s - log_s.rolling(30).mean()

print('Box-Cox lambda:', lam)


In [ ]:
fig, ax = plt.subplots(3, 2, figsize=(12, 9))

close.plot(ax=ax[0, 0], title='Original')
log_s.plot(ax=ax[0, 1], title='Log')
sq_s.plot(ax=ax[1, 0], title='Square')
bc_s.plot(ax=ax[1, 1], title='Box-Cox')
log_diff.plot(ax=ax[2, 0], title='Log + Differencing')
detrended.dropna().plot(ax=ax[2, 1], title='Detrended (Log)')

plt.tight_layout()
plt.show()


In [ ]:
transformed_series = {
    'Original': close,
    'Log': log_s,
    'Square': sq_s,
    'Box-Cox': bc_s,
    'Differencing': diff_s,
    'Log + Differencing': log_diff,
    'Detrended': detrended.dropna()
}

for name, series in transformed_series.items():
    result = adfuller(series.dropna())
    print(name, 'ADF p-value:', result[1])


## Experiment 7


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(close.dropna(), lags=40, ax=ax[0])
plot_pacf(close.dropna(), lags=40, ax=ax[1], method='ywm')
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(log_diff.dropna(), lags=40, ax=ax[0])
plot_pacf(log_diff.dropna(), lags=40, ax=ax[1], method='ywm')
plt.tight_layout()
plt.show()


## Experiment 8


In [ ]:
y = log_diff.dropna()
n = int(len(y) * 0.8)
train, test = y.iloc[:n], y.iloc[n:]


In [ ]:
ar_model = ARIMA(train, order=(1, 0, 0)).fit()
ma_model = ARIMA(train, order=(0, 0, 1)).fit()

print('--- AR(1) Model Summary ---')
print(ar_model.summary())
print('--- MA(1) Model Summary ---')
print(ma_model.summary())


In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

ar_resid = ar_model.resid

ar_model.plot_diagnostics(figsize=(12, 8))
plt.show()

fig, ax = plt.subplots(figsize=(8, 3))
plot_acf(ar_resid.dropna(), lags=30, ax=ax)
plt.show()

print(acorr_ljungbox(ar_resid.dropna(), lags=[10], return_df=True))


In [ ]:
ar_forecast = ar_model.forecast(steps=len(test))
ma_forecast = ma_model.forecast(steps=len(test))

ar_rmse = np.sqrt(mean_squared_error(test, ar_forecast))
ma_rmse = np.sqrt(mean_squared_error(test, ma_forecast))
ar_mae = mean_absolute_error(test, ar_forecast)
ma_mae = mean_absolute_error(test, ma_forecast)

print('AR(1) RMSE:', ar_rmse)
print('MA(1) RMSE:', ma_rmse)
print('AR(1) MAE:', ar_mae)
print('MA(1) MAE:', ma_mae)

plt.figure(figsize=(10, 4))
plt.plot(test.index, test.values, label='Actual')
plt.plot(test.index, np.asarray(ar_forecast), label='AR(1) Forecast')
plt.plot(test.index, np.asarray(ma_forecast), label='MA(1) Forecast')
plt.legend()
plt.title('AR and MA Forecast vs Actual')
plt.show()

future_forecast = ar_model.forecast(steps=30)
print('Next 30-step AR forecast:')
print(future_forecast)


## Experiment 9


In [ ]:
n = int(len(log_s) * 0.9)
train_log = log_s.iloc[:n]
test_log = log_s.iloc[n:]

results = []
for p, q in itertools.product(range(4), range(4)):
    try:
        fitted = ARIMA(train_log, order=(p, 1, q)).fit()
        results.append(((p, 1, q), fitted.aic))
    except Exception:
        pass

best_order, best_aic = min(results, key=lambda x: x[1])
print('Best order and AIC:', best_order, best_aic)


In [ ]:
best_model = ARIMA(train_log, order=best_order).fit()
print(best_model.summary())

best_model.plot_diagnostics(figsize=(12, 8))
plt.show()


In [ ]:
forecast_result = best_model.get_forecast(steps=len(test_log))
forecast_log = forecast_result.predicted_mean
conf_int_log = forecast_result.conf_int()

actual_price = np.exp(test_log)
forecast_price = np.exp(forecast_log)
lower_price = np.exp(conf_int_log.iloc[:, 0])
upper_price = np.exp(conf_int_log.iloc[:, 1])

plt.figure(figsize=(10, 4))
plt.plot(close.index[:n], close.iloc[:n].values, label='Train')
plt.plot(actual_price.index, actual_price.values, label='Actual')
plt.plot(forecast_price.index, forecast_price.values, label='Forecast')
plt.fill_between(
    forecast_price.index,
    lower_price.values,
    upper_price.values,
    alpha=0.2
)
plt.legend()
plt.title('ARIMA Price Forecast vs Actual')
plt.show()

print('RMSE:', np.sqrt(mean_squared_error(actual_price, forecast_price)))


## Experiment 10


In [ ]:
log_diff_clean = log_diff.reset_index(drop=True)

arma = ARIMA(log_diff_clean, order=(1, 0, 1)).fit()
print(arma.summary())
print('Next 10-step Forecast (Log Returns):')
print(arma.forecast(steps=10))


In [ ]:
monthly_log = np.log(monthly.dropna())
n = int(len(monthly_log) * 0.9)
train_monthly = monthly_log.iloc[:n]
test_monthly = monthly_log.iloc[n:]

sar = SARIMAX(
    train_monthly,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

print(sar.summary())
sar.plot_diagnostics(figsize=(12, 8))
plt.show()


In [ ]:
sar_forecast = sar.get_forecast(steps=12)
forecast_mean = np.exp(sar_forecast.predicted_mean)
forecast_ci = sar_forecast.conf_int()
lower_ci = np.exp(forecast_ci.iloc[:, 0])
upper_ci = np.exp(forecast_ci.iloc[:, 1])

plt.figure(figsize=(10, 4))
plt.plot(np.exp(monthly_log), label='Actual')
plt.plot(forecast_mean.index, forecast_mean.values, label='SARIMA Forecast')
plt.fill_between(
    forecast_mean.index,
    lower_ci.values,
    upper_ci.values,
    alpha=0.2
)
plt.legend()
plt.title('SARIMA 12-Month Forecast')
plt.show()

print('Next 12 months forecast:')
print(forecast_mean)
